# Phase 3 - Rhetorical roles with Arch 2 - Encoder-decoder (encoder used for tagging)

Starts from the Phase-1 model (T5-style encoder-decoder (109.5M); only its 6-layer bidirectional encoder is needed for tagging.) and labels every sentence of a judgment with one of 13 rhetorical
roles. Primary metric: macro-F1. CL and IT documents are trained together; a domain embedding tells them apart and
scores are reported per domain.

**The model**
1. The Phase-1 backbone reads *packs* of consecutive sentences (up to 512 tokens), so each sentence is encoded next to
   its neighbours (Cohan et al. 2019, *Pretrained LMs for sequential sentence classification*). For the decoder-only
   models the causal mask is lifted (Dukic & Snajder 2024; LLM2Vec). Each sentence is mean-pooled.
2. Relative-position and domain embeddings are added: the role depends strongly on where a sentence sits.
3. A BiLSTM reads the whole document's sentence vectors, then a CRF picks the label sequence - roles come in long blocks
   (94% of CL and 75% of IT sentences repeat the previous label). (Bhattacharya et al. 2019; Kalamkar et al. 2022.)
4. Auxiliary *label-shift* head: "does the role change at this sentence?" (Malik et al. 2022, the paper behind this dataset).
5. Macro-F1 rewards rare roles (Issue, None, RulingByPresentCourt...): class-weighted loss, and at the end a prior
   correction (logit adjustment, Menon et al. 2021) whose strength - and CRF vs argmax - is chosen on the dev set.

**Experiments** (each one is its own submission)

| Experiment | Trained parameters |
|---|---|
| `full` | everything, with layer-wise LR decay - strong reference and the distillation teacher |
| `lorma` | **LoRMA** (compulsory): `(alpha/r BA + I) W x` on the attention projections (Bihany, Patel & Modi 2025) |
| `lora` | LoRA on the same projections (Hu et al. 2021) |
| `ia3` | IA3: learned rescaling of keys/values and feed-forward activations (Liu et al. 2022) |
| `prompt` | prompt tuning: 16 learned virtual tokens in front of every pack (Lester et al. 2021) |
| `kd` | distillation: the fine-tuned `full` model teaches a half-depth student through soft labels **and** its sentence representations (Hinton et al. 2015) |

In every PEFT run the backbone is frozen; only the adapter parameters and the task head (BiLSTM, CRF, embeddings) train.

**Kaggle settings:** GPU T4 x2, Internet on, secret `HF` holding your Hugging Face token. Then Run All.

## 1. Setup

In [ ]:
import os, sys, json, glob, shutil, time
import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

WORK_DIR = "/kaggle/working/phase3_arch2"
os.makedirs(WORK_DIR, exist_ok=True)
sys.path.insert(0, WORK_DIR)

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. Configuration
Everything you might want to change lives here.

In [ ]:
%%writefile /kaggle/working/phase3_arch2/config.py
"""Every knob of this Phase-3 (rhetorical role) notebook."""
import os
import json

ARCH = "arch2"
WORK_DIR = f"/kaggle/working/phase3_{ARCH}"
DATA_DIR = os.path.join(WORK_DIR, "data")
CACHE_DIR = os.path.join(WORK_DIR, "phase1")        # Phase-1 files downloaded from Hugging Face land here
OUT_DIR = os.path.join(WORK_DIR, "runs")             # one folder per experiment

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
_P3 = os.path.join(DATA_DIR, "Phase_3_RR")
FILES = {f"{dom}_{split}": os.path.join(_P3, "Test" if split == "test" else "Train", f"{dom}_{split}.json")
         for dom in ("CL", "IT") for split in ("train", "dev", "test")}

# ------------------------------------------------------------------ the Phase-1 model
# Looked up in this order: an attached Kaggle dataset whose path mentions "arch2", then these HF repos.
PHASE1_REPOS = ["D-240357/legal-arch2-encdec"]
WEIGHT_FILES = ["model_final.pt", "model_best.pt"]
TOKENIZER_FILES = ["tokenizer.json", "data/tokenizer.json"]
TOKENIZER_REPOS = ["D-240357/legal-arch2-encdec", "D-240357/legal-arch3-decoder"]   # same BPE file as arch 3
KAGGLE_HINT = "arch2"

RR_REPO = "D-240357/legal-rr-arch2"             # fine-tuned checkpoints, results and submission zips go here

# ------------------------------------------------------------------ experiments
# Each one is a separate submission. "full" is the strong reference and the distillation teacher;
# "kd" waits for it to finish.
METHODS = ["full", "lorma", "lora", "ia3", "prompt", "kd"]
KD_TEACHER = "full"

# ------------------------------------------------------------------ input / context
MAX_SENT_TOKENS = 96           # longer sentences are cut (median sentence is ~40 tokens)
PACK_TOKENS = 512              # consecutive sentences share one sequence, so each one is read in context
PACK_CONTEXT = True            # False = every sentence encoded alone (the "sentence-only" ablation)
DOC_CONTEXT = "bilstm"         # "none" = no document-level layer (another ablation)
BIDIRECTIONAL = True            # the encoder is bidirectional anyway
SPAN_SENTS = 96               # consecutive sentences per training example (whole documents at test time)
TRAIN_ON_DEV = False           # True for a final run on train+dev once the settings are chosen
TRAIN_DOC_LIMIT = None         # set to e.g. 8 for a quick end-to-end smoke test

# ------------------------------------------------------------------ head and losses
CONTEXT_DIM = 512
HEAD_DROPOUT = 0.2
CE_WEIGHT = 0.5                # class-weighted cross-entropy next to the CRF loss (helps the rare roles)
CLASS_WEIGHT_POWER = 0.5       # weight = (1 / frequency) ** power
SHIFT_WEIGHT = 0.3             # auxiliary "does the role change here?" loss (Malik et al. 2022)
TAU_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]   # prior correction tried on dev at the end (Menon et al. 2021)

# ------------------------------------------------------------------ optimisation
EPOCHS = 20                    # Phase-2 runs peaked in their last epochs, and one epoch here is only ~1-2 min
PATIENCE = 5
GRAD_ACCUM = 2
EVAL_PACK_BATCH = 16
WARMUP_FRAC = 0.1
WEIGHT_DECAY = 0.01
HEAD_LR = 1e-3
FULL_LR = 1e-4                 # top layer; each layer below gets LLRD times the one above it
                               # (in Phase 2 a higher LR on the top layers beat a timid full fine-tune)
LLRD = 0.85
ADAPTER_LR = 5e-4
ADAPTER_RANK = 8
LORA_ALPHA = 16
LORMA_ALPHA = 8
ADAPTER_DROPOUT = 0.05
ADAPTER_TARGETS = ["qkv", "proj"]
IA3_OUT_TARGETS = ["qkv"]       # key/value projections
IA3_IN_TARGETS = ["down"]         # feed-forward down projections
PROMPT_LEN = 16
PROMPT_LR = 1e-2

KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5                 # weight of the task loss; the rest goes to matching the teacher
KD_HIDDEN_WEIGHT = 1.0         # also match the teacher's sentence representations
STUDENT_LAYERS = [0, 2, 4]

SEED = 13

_override = os.environ.get("P3_CFG_OVERRIDE")
if _override:
    globals().update(json.loads(_override))

## 3. Code
The Phase-1 tokenizer and model (unchanged, so the trained weights load), plus the rhetorical-role pieces.

In [ ]:
%%writefile /kaggle/working/phase3_arch2/tokenizer.py
"""
Byte-level BPE tokenizer, written from scratch (stdlib only for the tokenizer logic).

How it works, in short:
  1. A regex chops text into "pre-tokens" (a word with its leading space, a run of digits,
     a run of punctuation, ...). Merges never cross these boundaries.
  2. Every pre-token is turned into raw UTF-8 bytes (ids 0..255), so *any* text can be
     encoded - there is no <unk> and no lowercasing.
  3. Training repeatedly merges the most frequent adjacent id pair into a new id.
     We train on the table of *unique* pre-tokens with their counts, which is what keeps
     it fast in pure Python.
  4. Encoding replays the learned merges (lowest rank first) per pre-token, and caches the
     result, so repeated words cost a single dict lookup.
"""
import re
import json
import heapq
from collections import Counter, defaultdict

# Close to the GPT-2 splitter, but built on plain `re` (no \p{L}), so it needs no extra package.
PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+",
    re.UNICODE,
)

EOS_TOKEN = "<|endoftext|>"
PAD_TOKEN = "<|pad|>"


class ByteBPE:
    def __init__(self, merges=None, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = [tuple(m) for m in (merges or [])]      # merge i creates id 256 + i
        self.eos_id = vocab_size - 2
        self.pad_id = vocab_size - 1
        self._cache = {}
        self._rebuild_tables()

    # ------------------------------------------------------------- internals
    def _rebuild_tables(self):
        self.ranks = {pair: (i, 256 + i) for i, pair in enumerate(self.merges)}
        vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            vocab.append(vocab[a] + vocab[b])
        self.vocab = vocab
        self._cache.clear()

    # -------------------------------------------------------------- training
    @classmethod
    def train(cls, texts, vocab_size=32000, log_every=2000):
        """Learn merges from an iterable of strings."""
        n_merges = vocab_size - 256 - 2          # leave room for the two special tokens
        word_counts = Counter()
        for text in texts:
            word_counts.update(PRETOKEN_RE.findall(text))
        print(f"  unique pre-tokens: {len(word_counts):,}")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_count = defaultdict(int)
        where = defaultdict(set)                 # pair -> indices of words that (may) contain it
        for wi, w in enumerate(words):
            f = freqs[wi]
            for pair in zip(w, w[1:]):
                pair_count[pair] += f
                where[pair].add(wi)

        # Max-heap with lazy invalidation: stale entries are fixed up when they surface.
        heap = [(-c, p) for p, c in pair_count.items()]
        heapq.heapify(heap)

        merges = []
        vocab = [bytes([i]) for i in range(256)]
        for m in range(n_merges):
            best = None
            while heap:
                neg, pair = heapq.heappop(heap)
                real = pair_count.get(pair, 0)
                if real <= 0:
                    continue
                if real != -neg:
                    heapq.heappush(heap, (-real, pair))
                    continue
                best = pair
                break
            if best is None:
                print(f"  ran out of pairs after {m} merges")
                break

            new_id = 256 + m
            merges.append(best)
            a, b = best
            vocab.append(vocab[a] + vocab[b])
            touched = {}
            for wi in where.pop(best, ()):
                w = words[wi]
                f = freqs[wi]
                # find the occurrences in this word (left to right, non-overlapping)
                if len(w) < 2:
                    continue
                i, out, changed = 0, [], False
                while i < len(w):
                    if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                        changed = True
                    else:
                        out.append(w[i])
                        i += 1
                if not changed:
                    continue                      # stale entry in `where`
                for pair in zip(w, w[1:]):        # take the old word's pairs away ...
                    pair_count[pair] -= f
                for pair in zip(out, out[1:]):    # ... and add the new word's pairs
                    pair_count[pair] += f
                    where[pair].add(wi)
                    touched[pair] = pair_count[pair]
                words[wi] = out
            for pair, c in touched.items():
                if c > 0:
                    heapq.heappush(heap, (-c, pair))
            pair_count.pop(best, None)

            if (m + 1) % log_every == 0:
                print(f"  merge {m + 1:,}/{n_merges:,}  last = {vocab[-1]!r}")

        return cls(merges, vocab_size)

    # -------------------------------------------------------------- encoding
    def _encode_word(self, word):
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) > 1:
            best_rank, best_pair, best_new = None, None, None
            for pair in zip(ids, ids[1:]):
                hit = ranks.get(pair)
                if hit is not None and (best_rank is None or hit[0] < best_rank):
                    best_rank, best_pair, best_new = hit[0], pair, hit[1]
            if best_pair is None:
                break
            a, b = best_pair
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(best_new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out
        return ids

    def encode(self, text):
        cache = self._cache
        if len(cache) > 1_000_000:               # keep RAM bounded on huge corpora
            cache.clear()
        out = []
        for word in PRETOKEN_RE.findall(text):
            ids = cache.get(word)
            if ids is None:
                ids = cache[word] = self._encode_word(word)
            out.extend(ids)
        return out

    def decode(self, ids):
        specials = (self.eos_id, self.pad_id)
        data = b"".join(self.vocab[i] for i in ids if i < len(self.vocab) and i not in specials)
        return data.decode("utf-8", errors="replace")

    # ----------------------------------------------------------- persistence
    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges,
                       "eos": EOS_TOKEN, "pad": PAD_TOKEN}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            blob = json.load(f)
        return cls(blob["merges"], blob["vocab_size"])


# ---------------------------------------------------------------------------------------------------
# Phase 2 addition: encode with character offsets, so predicted tokens can be mapped back to exact spans.
# ---------------------------------------------------------------------------------------------------
# Same splitter as above, except a run of punctuation is cut into single characters. Gold entities often end
# in the middle of runs like "Ors.[1]" or "Cr.P.C.,"; with this split 99.3% of gold spans start and end on a
# token boundary (98.1% without it). Every piece is still encoded with the Phase-1 merges.
NER_PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]|\s+(?!\S)|\s+",
    re.UNICODE,
)


class NerTokenizer(ByteBPE):
    split_punct = True

    def encode_with_offsets(self, text):
        """Returns ids and, per token, the [start, end) of its non-whitespace characters in `text`."""
        splitter = NER_PRETOKEN_RE if self.split_punct else PRETOKEN_RE
        ids, starts, ends = [], [], []
        for m in splitter.finditer(text):
            word, base = m.group(), m.start()
            toks = self._cache.get(word)
            if toks is None:
                toks = self._cache[word] = self._encode_word(word)
            # byte position -> character position inside this piece (pieces can contain multi-byte characters)
            char_at_byte = []
            for ci, ch in enumerate(word):
                char_at_byte += [ci] * len(ch.encode("utf-8"))
            b = 0
            for t in toks:
                n = len(self.vocab[t])
                cs, ce = char_at_byte[b], char_at_byte[b + n - 1] + 1
                b += n
                piece = word[cs:ce]
                s = base + cs + (len(piece) - len(piece.lstrip()))
                e = base + ce - (len(piece) - len(piece.rstrip()))
                if s >= e:
                    s = e = base + ce
                ids.append(t)
                starts.append(s)
                ends.append(e)
        return ids, starts, ends

In [ ]:
%%writefile /kaggle/working/phase3_arch2/phase1_model.py
"""
Phase-1 model code, unchanged except for the small switches marked "Phase 2", so the saved weights load as-is.
"""
"""
Arch 2: a compact encoder-decoder (T5-style) language model, written from scratch.

It is shaped like the task itself: the ENCODER reads the 512-word prompt with full bidirectional
attention, and the DECODER writes the continuation, attending to the prompt through cross-attention.

Same efficiency toolbox as the other architectures:
  * RMSNorm (pre-norm), no biases, SwiGLU feed-forward
  * RoPE inside self-attention (cross-attention needs none: the encoder output already carries position)
  * grouped-query attention everywhere (fewer KV heads than query heads)
  * everything runs through F.scaled_dot_product_attention (fused kernels, no T x T matrix)
  * one embedding table shared by encoder input, decoder input and the output head
  * KV-cache decoding with the cross-attention K/V computed once per prompt
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    vocab_size: int = 32000
    d_model: int = 768
    n_head: int = 12
    n_kv_head: int = 4
    n_enc_layer: int = 6
    n_dec_layer: int = 6
    d_ff: int = 2048
    enc_len: int = 1024            # prompt tokens the encoder sees
    dec_len: int = 320             # continuation tokens the decoder is trained on
    rope_base: float = 10000.0
    norm_eps: float = 1e-5


class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32 * self.weight).to(x.dtype)


def build_rope_cache(head_dim, max_len, base):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = torch.outer(torch.arange(max_len).float(), inv_freq)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(x.dtype)


def repeat_kv(x, group):
    return x if group == 1 else x.repeat_interleave(group, dim=1)


class SelfAttention(nn.Module):
    """Grouped-query self-attention with RoPE. `causal` picks decoder (True) or encoder (False) behaviour."""

    def __init__(self, cfg, causal):
        super().__init__()
        self.causal = causal
        self.n_head, self.n_kv = cfg.n_head, cfg.n_kv_head
        self.head_dim = cfg.d_model // cfg.n_head
        self.group = cfg.n_head // cfg.n_kv_head
        self.q_dim = self.n_head * self.head_dim
        self.kv_dim = self.n_kv * self.head_dim
        self.qkv = nn.Linear(cfg.d_model, self.q_dim + 2 * self.kv_dim, bias=False)
        self.proj = nn.Linear(self.q_dim, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, past=None, key_mask=None):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).split([self.q_dim, self.kv_dim, self.kv_dim], dim=-1)
        q = apply_rope(q.view(B, T, self.n_head, self.head_dim).transpose(1, 2), cos, sin)
        k = apply_rope(k.view(B, T, self.n_kv, self.head_dim).transpose(1, 2), cos, sin)
        v = v.view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        if past is not None:
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        present = (k, v)
        k, v = repeat_kv(k, self.group), repeat_kv(v, self.group)

        if key_mask is not None:                       # padded prompt (inference) or single-step decoding
            o = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask)
        else:
            o = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and past is None and T > 1)
        return self.proj(o.transpose(1, 2).reshape(B, T, self.q_dim)), present


class CrossAttention(nn.Module):
    """Decoder queries attend to the encoder output. No positional rotation here."""

    def __init__(self, cfg):
        super().__init__()
        self.n_head, self.n_kv = cfg.n_head, cfg.n_kv_head
        self.head_dim = cfg.d_model // cfg.n_head
        self.group = cfg.n_head // cfg.n_kv_head
        self.q_dim = self.n_head * self.head_dim
        self.kv_dim = self.n_kv * self.head_dim
        self.q = nn.Linear(cfg.d_model, self.q_dim, bias=False)
        self.kv = nn.Linear(cfg.d_model, 2 * self.kv_dim, bias=False)
        self.proj = nn.Linear(self.q_dim, cfg.d_model, bias=False)

    def project_kv(self, enc_out):
        B, S, _ = enc_out.shape
        k, v = self.kv(enc_out).split([self.kv_dim, self.kv_dim], dim=-1)
        return (k.view(B, S, self.n_kv, self.head_dim).transpose(1, 2),
                v.view(B, S, self.n_kv, self.head_dim).transpose(1, 2))

    def forward(self, x, enc_out=None, cross_kv=None, enc_mask=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k, v = cross_kv if cross_kv is not None else self.project_kv(enc_out)
        k, v = repeat_kv(k, self.group), repeat_kv(v, self.group)
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=enc_mask)
        return self.proj(o.transpose(1, 2).reshape(B, T, self.q_dim))


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.gate_up = nn.Linear(d_model, 2 * hidden, bias=False)
        self.down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        gate, up = self.gate_up(x).chunk(2, dim=-1)
        return self.down(F.silu(gate) * up)


class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = SelfAttention(cfg, causal=False)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff)

    def forward(self, x, cos, sin, key_mask=None):
        a, _ = self.attn(self.attn_norm(x), cos, sin, key_mask=key_mask)
        x = x + a
        return x + self.ffn(self.ffn_norm(x))


class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.self_attn = SelfAttention(cfg, causal=True)
        self.cross_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.cross_attn = CrossAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff)

    def forward(self, x, cos, sin, enc_out=None, cross_kv=None, enc_mask=None, past=None, key_mask=None):
        a, present = self.self_attn(self.self_norm(x), cos, sin, past=past, key_mask=key_mask)
        x = x + a
        x = x + self.cross_attn(self.cross_norm(x), enc_out=enc_out, cross_kv=cross_kv, enc_mask=enc_mask)
        return x + self.ffn(self.ffn_norm(x)), present


class EncDecLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)          # shared by encoder, decoder and output head
        self.enc_layers = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.n_enc_layer)])
        self.enc_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.dec_layers = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.n_dec_layer)])
        self.dec_norm = RMSNorm(cfg.d_model, cfg.norm_eps)

        cos, sin = build_rope_cache(cfg.d_model // cfg.n_head, max(cfg.enc_len, 2 * cfg.dec_len), cfg.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        n_res = cfg.n_enc_layer + 2 * cfg.n_dec_layer                      # residual writers: shrink them with depth
        scale = 0.02 / math.sqrt(n_res)
        for layer in self.enc_layers:
            nn.init.normal_(layer.attn.proj.weight, std=scale)
            nn.init.normal_(layer.ffn.down.weight, std=scale)
        for layer in self.dec_layers:
            nn.init.normal_(layer.self_attn.proj.weight, std=scale)
            nn.init.normal_(layer.cross_attn.proj.weight, std=scale)
            nn.init.normal_(layer.ffn.down.weight, std=scale)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def active_params(self):                                              # dense model: everything is active
        return self.num_params()

    # ---- pieces ------------------------------------------------------------
    def encode(self, enc_ids, enc_mask=None):
        """enc_mask: optional (B, S) bool, True for real tokens (only needed for padded batches)."""
        B, S = enc_ids.shape
        cos, sin = self.rope_cos[:S], self.rope_sin[:S]
        key_mask = None if enc_mask is None else enc_mask[:, None, None, :]
        x = self.tok_emb(enc_ids)
        for layer in self.enc_layers:
            x = layer(x, cos, sin, key_mask)
        return self.enc_norm(x)

    def hidden_states(self, idx):
        """Encoder states (B, T, d_model): what a Phase 2-7 task head sits on top of."""
        return self.encode(idx)

    def cross_kv(self, enc_out):
        return [layer.cross_attn.project_kv(enc_out) for layer in self.dec_layers]

    def decode(self, dec_ids, enc_out=None, cross_kvs=None, enc_mask=None, caches=None, start_pos=0, last_only=False):
        B, T = dec_ids.shape
        cos, sin = self.rope_cos[start_pos:start_pos + T], self.rope_sin[start_pos:start_pos + T]
        cross_mask = None if enc_mask is None else enc_mask[:, None, None, :]
        x = self.tok_emb(dec_ids)
        new_caches = []
        for i, layer in enumerate(self.dec_layers):
            x, present = layer(x, cos, sin, enc_out=enc_out,
                               cross_kv=None if cross_kvs is None else cross_kvs[i], enc_mask=cross_mask,
                               past=None if caches is None else caches[i])
            new_caches.append(present)
        x = self.dec_norm(x)
        if last_only:
            x = x[:, -1:, :]
        return F.linear(x, self.tok_emb.weight), new_caches

    # ---- training forward --------------------------------------------------
    def forward(self, enc_ids, dec_ids, targets):
        """Teacher-forced: returns (cross-entropy over the decoder tokens, z-loss)."""
        enc_out = self.encode(enc_ids)
        logits, _ = self.decode(dec_ids, enc_out=enc_out)
        logits = logits.float()
        ce = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        z = torch.logsumexp(logits, dim=-1).pow(2).mean()
        return ce, z


# ---------------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------------
def _sample(logits, temperature, top_k, top_p):
    logits = logits / max(temperature, 1e-5)
    k = min(top_k, logits.size(-1)) if top_k > 0 else logits.size(-1)
    vals, idx = torch.topk(logits, k, dim=-1)
    probs = F.softmax(vals, dim=-1)
    if top_p < 1.0:
        before = probs.cumsum(-1) - probs
        vals = vals.masked_fill(before >= top_p, float("-inf"))
        probs = F.softmax(vals, dim=-1)
    return idx.gather(-1, torch.multinomial(probs, 1)).squeeze(-1)


@torch.no_grad()
def generate(model, prompts, tokenizer, max_new_tokens=320, target_words=128,
             temperature=0.7, top_k=50, top_p=0.92, check_every=8, suppress_eos=True):
    """
    prompts: list of token-id lists (<= enc_len each). The prompts are left-padded into one batch and the
    padding is masked out of both the encoder and the cross-attention. Returns one continuation string each.
    """
    model.eval()
    dev = next(model.parameters()).device
    B = len(prompts)
    S = max(len(p) for p in prompts)
    assert S <= model.cfg.enc_len, "prompt longer than the encoder context"
    assert max_new_tokens <= 2 * model.cfg.dec_len, "continuation longer than the cached RoPE table"
    pad, eos = tokenizer.pad_id, tokenizer.eos_id

    ids = torch.full((B, S), pad, dtype=torch.long)
    valid = torch.zeros((B, S), dtype=torch.bool)
    for r, p in enumerate(prompts):
        ids[r, S - len(p):] = torch.tensor(p, dtype=torch.long)
        valid[r, S - len(p):] = True
    ids, valid = ids.to(dev), valid.to(dev)

    enc_out = model.encode(ids, valid)
    cross_kvs = model.cross_kv(enc_out)

    cur = torch.full((B, 1), eos, dtype=torch.long, device=dev)            # decoder start token
    out = torch.empty((B, 0), dtype=torch.long, device=dev)
    finished = torch.zeros(B, dtype=torch.bool, device=dev)
    caches = None
    for step in range(max_new_tokens):
        logits, caches = model.decode(cur, cross_kvs=cross_kvs, enc_mask=valid, caches=caches,
                                      start_pos=step, last_only=True)
        step_logits = logits[:, -1].float()
        step_logits[:, pad] = float("-inf")
        if suppress_eos:
            step_logits[:, eos] = float("-inf")
        nxt = _sample(step_logits, temperature, top_k, top_p)
        nxt = torch.where(finished, torch.full_like(nxt, pad), nxt)
        out = torch.cat([out, nxt[:, None]], dim=1)
        finished |= nxt == eos
        cur = nxt[:, None]

        if (step + 1) % check_every == 0:
            for r in torch.nonzero(~finished).flatten().tolist():
                if len(tokenizer.decode(out[r].tolist()).split()) >= target_words:
                    finished[r] = True
        if bool(finished.all()):
            break

    texts = []
    for r in range(B):
        toks = out[r].tolist()
        if eos in toks:
            toks = toks[:toks.index(eos)]
        toks = [t for t in toks if t != pad]
        texts.append(" ".join(tokenizer.decode(toks).split()[:target_words]))
    return texts

In [ ]:
%%writefile /kaggle/working/phase3_arch2/hub_files.py
"""Finds Phase-1 files: an attached Kaggle dataset first, then a local cache, then Hugging Face."""
import os
import re

from huggingface_hub import hf_hub_download

KAGGLE_INPUT = "/kaggle/input"


def _squash(text):
    return re.sub(r"[^a-z0-9]", "", text.lower())


def find_in_kaggle_input(filename, hint):
    """Only accepts a file whose path mentions the architecture (e.g. 'arch3', 'arch-3', 'Arch_3'),
    because arch 2, 3 and 4 all call their weights model_final.pt."""
    if not os.path.isdir(KAGGLE_INPUT):
        return None
    hits = []
    for root, _, files in os.walk(KAGGLE_INPUT):
        if filename in files:
            path = os.path.join(root, filename)
            if _squash(hint) in _squash(path):
                hits.append(path)
    return sorted(hits)[0] if hits else None


def fetch_from_hub(repo, filename, local_dir):
    try:
        return hf_hub_download(repo, filename, local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def locate(filenames, repos, hint, local_dir):
    """Names are tried in order of preference. Returns (path, where it came from)."""
    for name in filenames:
        path = find_in_kaggle_input(os.path.basename(name), hint)
        if path:
            return path, "kaggle input"
    for name in filenames:
        for repo in repos:
            cached = os.path.join(local_dir, _squash(repo), name)
            if os.path.exists(cached):
                return cached, f"cache of hf://{repo}"
            path = fetch_from_hub(repo, name, os.path.join(local_dir, _squash(repo)))
            if path:
                return path, f"hf://{repo}"
    raise FileNotFoundError(f"None of {filenames} found under /kaggle/input (path must mention '{hint}') or in "
                            f"{repos}. Attach the Phase-1 weights as a Kaggle dataset, or check HF_TOKEN and the repo names.")

In [ ]:
%%writefile /kaggle/working/phase3_arch2/backbone.py
"""Loads the Phase-1 arch 2 weights (encoder-decoder) and uses its bidirectional ENCODER for rhetorical roles."""
import os

import torch
import torch.nn as nn

import config as C
from hub_files import locate, fetch_from_hub  # noqa: F401  (fetch_from_hub is used by run_rr)
from phase1_model import EncDecLM, ModelConfig
from tokenizer import NerTokenizer


def add_prompt(x, mask, prompt):
    """Prompt tuning (Lester et al. 2021): prepend learned vectors to the token embeddings."""
    if prompt is None:
        return x, mask, 0
    P = prompt.size(0)
    x = torch.cat([prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1), x], dim=1)
    mask = torch.cat([mask.new_ones(mask.size(0), P), mask], dim=1)
    return x, mask, P


def load_tokenizer():
    path, _ = locate(C.TOKENIZER_FILES, C.TOKENIZER_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    tok = NerTokenizer.load(path)
    tok.sep_id = tok.eos_id               # marks the start of every sentence inside a pack
    return tok


class Backbone(nn.Module):
    """The decoder is dropped: tagging only needs the encoder, which already attends both ways."""

    def __init__(self, model, keep_layers):
        super().__init__()
        self.d_model = model.cfg.d_model
        self.tok_emb = model.tok_emb
        layers = model.enc_layers if keep_layers is None else [model.enc_layers[i] for i in keep_layers]
        self.layers = nn.ModuleList(layers)
        self.final_norm = model.enc_norm
        self.register_buffer("rope_cos", model.rope_cos.clone(), persistent=False)
        self.register_buffer("rope_sin", model.rope_sin.clone(), persistent=False)

    def forward(self, ids, mask, prompt=None):
        x, mask, P = add_prompt(self.tok_emb(ids), mask, prompt)
        T = x.size(1)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        key_mask = None if bool(mask.all()) else mask[:, None, None, :]
        for layer in self.layers:
            x = layer(x, cos, sin, key_mask)
        return self.final_norm(x)[:, P:]


def build_backbone(keep_layers=None, bidirectional=True):
    path, source = locate(C.WEIGHT_FILES, C.PHASE1_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    blob = torch.load(path, map_location="cpu")
    model = EncDecLM(ModelConfig(**blob["model_cfg"]))
    model.load_state_dict({k: v.float() for k, v in blob["model"].items()})
    if os.environ.get("P2_VERBOSE", "1") == "1":
        print(f"Phase-1 arch 2 weights: {path} ({source})", flush=True)
    return Backbone(model, keep_layers)


def student_layers():
    return list(C.STUDENT_LAYERS)

In [ ]:
%%writefile /kaggle/working/phase3_arch2/rr_core.py
"""
Phase 3 (rhetorical roles) machinery shared by all four architectures.

Model, bottom to top:
  1. the Phase-1 backbone reads packs of consecutive sentences (up to PACK_TOKENS), so every sentence is encoded
     next to its neighbours (Cohan et al. 2019), and each sentence is mean-pooled;
  2. relative position + domain (CL / IT) embeddings are added;
  3. a BiLSTM runs over the whole document's sentence vectors;
  4. a linear-chain CRF picks the label sequence (roles come in long blocks: 94% of CL and 75% of IT sentences
     share the previous sentence's label);
  5. an auxiliary head predicts whether the role changes at each sentence (label-shift prediction, Malik et al. 2022).
"""
import json
import math
import random
import zipfile
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

LABELS = ["Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent", "PrecedentReliedUpon",
          "PrecedentNotReliedUpon", "PrecedentOverruled", "RulingByLowerCourt", "RatioOfTheDecision",
          "RulingByPresentCourt", "None"]
LABEL_ID = {l: i for i, l in enumerate(LABELS)}
N_LABELS = len(LABELS)
DOMAINS = ["CL", "IT"]
N_POS_BUCKETS = 20


# --------------------------------------------------------------------------------------------- data
def load_split(path, domain):
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    docs = []
    if isinstance(raw, dict):                                   # test files: {doc_id: [sentences]}
        raw = [{"id": k, "text": v} for k, v in raw.items()]
    for d in raw:
        labels = [LABEL_ID[l] for l in d["labels"]] if "labels" in d else None
        docs.append({"id": str(d["id"]), "domain": DOMAINS.index(domain), "sents": list(d["text"]), "labels": labels})
    return docs


def tokenize_docs(docs, tok, max_sent_tokens):
    """Every sentence becomes [separator] + its first max_sent_tokens tokens."""
    for d in docs:
        d["tok"] = [[tok.sep_id] + tok.encode(s)[:max_sent_tokens] for s in d["sents"]]
    return docs


def pack(sent_tokens, pack_tokens):
    """Greedily put consecutive sentences into sequences of at most pack_tokens tokens.
    Returns the packs and, for every sentence, (pack index, first token, last token + 1)."""
    packs, where, cur = [], [], []
    for toks in sent_tokens:
        if cur and len(cur) + len(toks) > pack_tokens:
            packs.append(cur)
            cur = []
        where.append((len(packs), len(cur), len(cur) + len(toks)))
        cur = cur + toks
    if cur:
        packs.append(cur)
    return packs, where


def batch_packs(packs, where, pad_id, device, one_per_sentence=False):
    if one_per_sentence:                                        # sentence-only ablation: no neighbours in view
        packs = [packs[p][a:b] for p, a, b in where]
        where = [(i, 0, b - a) for i, (_, a, b) in enumerate(where)]
    L = max(len(p) for p in packs)
    ids = torch.full((len(packs), L), pad_id, dtype=torch.long)
    mask = torch.zeros((len(packs), L), dtype=torch.bool)
    sent_of_token = torch.full((len(packs), L), -1, dtype=torch.long)
    for i, p in enumerate(packs):
        ids[i, :len(p)] = torch.tensor(p)
        mask[i, :len(p)] = True
    for s, (p, a, b) in enumerate(where):
        sent_of_token[p, a:b] = s
    return ids.to(device), mask.to(device), sent_of_token.to(device)


def position_buckets(n):
    return torch.tensor([min(N_POS_BUCKETS - 1, int(N_POS_BUCKETS * i / n)) for i in range(n)], dtype=torch.long)


# ----------------------------------------------------------------------------------- adapters
class LoRALinear(nn.Module):
    """Hu et al. 2021: W x + (alpha / r) B A x, B = 0 at the start."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    """Bihany, Patel & Modi 2025 (LoRMA+): h = (alpha/r B A + I) W0 x, computed as W0x -> A -> B."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + F.linear(F.linear(self.drop(h), self.A), self.B) * self.scale
        return h + self.base.bias if self.base.bias is not None else h


class IA3Out(nn.Module):
    """Liu et al. 2022 (IA3): a learned vector rescales a projection's output (keys / values). Starts at 1."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.scale = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.scale


class IA3In(nn.Module):
    """IA3 on the feed-forward block: rescale the hidden activation just before the down projection."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.scale = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.scale)


def wrap_linears(root, names, make):
    swapped = 0
    for parent in list(root.modules()):
        for name, child in list(parent.named_children()):
            if name in names and isinstance(child, nn.Linear):
                setattr(parent, name, make(child))
                swapped += 1
    return swapped


# ------------------------------------------------------------------------------------------ CRF
class CRF(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.trans = nn.Parameter(torch.zeros(n, n))           # trans[prev, cur]
        self.start = nn.Parameter(torch.zeros(n))
        self.end = nn.Parameter(torch.zeros(n))

    def nll(self, em, tags):
        """Negative log-likelihood of one tag sequence. em: (T, n) emissions."""
        em = em.float()
        T = em.size(0)
        gold = self.start[tags[0]] + em[0, tags[0]] + self.end[tags[-1]]
        if T > 1:
            gold = gold + self.trans[tags[:-1], tags[1:]].sum() + em[torch.arange(1, T), tags[1:]].sum()
        alpha = self.start + em[0]
        for t in range(1, T):
            alpha = torch.logsumexp(alpha[:, None] + self.trans, dim=0) + em[t]
        return torch.logsumexp(alpha + self.end, dim=0) - gold

    @torch.no_grad()
    def viterbi(self, em):
        em = em.float().cpu().numpy()
        trans = self.trans.detach().float().cpu().numpy()
        score = self.start.detach().float().cpu().numpy() + em[0]
        back = np.zeros(em.shape, dtype=np.int64)
        for t in range(1, len(em)):
            cand = score[:, None] + trans
            back[t] = cand.argmax(0)
            score = cand.max(0) + em[t]
        score = score + self.end.detach().float().cpu().numpy()
        path = [int(score.argmax())]
        for t in range(len(em) - 1, 0, -1):
            path.append(int(back[t, path[-1]]))
        return path[::-1]


# ---------------------------------------------------------------------------------------- model
class RRModel(nn.Module):
    def __init__(self, backbone, cfg):
        super().__init__()
        self.backbone = backbone
        H = cfg.CONTEXT_DIM
        self.proj = nn.Sequential(nn.Linear(backbone.d_model, H), nn.LayerNorm(H))
        self.pos_emb = nn.Embedding(N_POS_BUCKETS, H)
        self.dom_emb = nn.Embedding(len(DOMAINS), H)
        self.drop = nn.Dropout(cfg.HEAD_DROPOUT)
        self.use_lstm = cfg.DOC_CONTEXT == "bilstm"
        self.lstm = nn.LSTM(H, H // 2, num_layers=2, bidirectional=True, batch_first=True, dropout=cfg.HEAD_DROPOUT)
        self.emit = nn.Linear(H, N_LABELS)
        self.shift = nn.Linear(2 * H, 1)
        self.crf = CRF(N_LABELS)
        self.prompt = None                      # (P, d_model) soft prompt, only for prompt tuning
        self.pack_tokens = cfg.PACK_TOKENS
        self.sentence_only = not cfg.PACK_CONTEXT

    def sentence_vectors(self, doc_tok, pad_id, device, pack_batch=None):
        """Encode a run of sentences (list of token lists) -> (S, d_model) mean-pooled sentence vectors."""
        packs, where = pack(doc_tok, self.pack_tokens)
        ids, mask, sot = batch_packs(packs, where, pad_id, device, self.sentence_only)
        S = len(doc_tok)
        out = torch.zeros(S, self.backbone.d_model, device=device)
        count = torch.zeros(S, 1, device=device)
        step = pack_batch or ids.size(0)
        for b in range(0, ids.size(0), step):
            h = self.backbone(ids[b:b + step], mask[b:b + step], prompt=self.prompt).float()
            s = sot[b:b + step].reshape(-1)
            keep = s >= 0
            out = out.index_add(0, s[keep], h.reshape(-1, h.size(-1))[keep])
            count = count.index_add(0, s[keep], torch.ones(int(keep.sum()), 1, device=device))
        return out / count.clamp(min=1)

    def head(self, sent_vecs, domain, n_total=None, offset=0):
        """Sentence vectors of a contiguous run -> (context states, emissions, shift logits)."""
        S = sent_vecs.size(0)
        n_total = n_total or S
        buckets = position_buckets(n_total)[offset:offset + S].to(sent_vecs.device)
        x = self.proj(sent_vecs) + self.pos_emb(buckets) + self.dom_emb.weight[domain]
        x = self.drop(x)
        if self.use_lstm:
            x, _ = self.lstm(x.unsqueeze(0))
            x = x.squeeze(0)
        ctx = self.drop(x)
        em = self.emit(ctx).float()
        shift = self.shift(torch.cat([ctx[:-1], ctx[1:]], dim=-1)).squeeze(-1).float() if S > 1 else em.new_zeros(0)
        return ctx, em, shift

    def forward(self, doc_tok, domain, pad_id, device, n_total=None, offset=0, pack_batch=None):
        vecs = self.sentence_vectors(doc_tok, pad_id, device, pack_batch)
        return self.head(vecs, domain, n_total, offset)


# ----------------------------------------------------------------------------- methods / params
def configure_method(model, method, cfg, tok_emb_weight):
    bb = model.backbone
    if method in ("full", "kd"):
        for p in model.parameters():
            p.requires_grad = True
        return "all weights trainable"

    for p in bb.parameters():
        p.requires_grad = False
    if method == "lora":
        n = wrap_linears(bb.layers, cfg.ADAPTER_TARGETS, lambda m: LoRALinear(m, cfg.ADAPTER_RANK, cfg.LORA_ALPHA, cfg.ADAPTER_DROPOUT))
        return f"LoRA on {n} projections (r={cfg.ADAPTER_RANK})"
    if method == "lorma":
        n = wrap_linears(bb.layers, cfg.ADAPTER_TARGETS, lambda m: LoRMALinear(m, cfg.ADAPTER_RANK, cfg.LORMA_ALPHA, cfg.ADAPTER_DROPOUT))
        return f"LoRMA on {n} projections (r={cfg.ADAPTER_RANK})"
    if method == "ia3":
        n_out = wrap_linears(bb.layers, cfg.IA3_OUT_TARGETS, IA3Out)
        n_in = wrap_linears(bb.layers, cfg.IA3_IN_TARGETS, IA3In)
        return f"IA3 vectors on {n_out} key/value projections and {n_in} feed-forward layers"
    if method == "prompt":
        # Lester et al. 2021: learned virtual tokens, initialised from real token embeddings
        rng = torch.Generator().manual_seed(cfg.SEED)
        pick = torch.randint(100, tok_emb_weight.size(0) - 100, (cfg.PROMPT_LEN,), generator=rng)
        model.prompt = nn.Parameter(tok_emb_weight[pick].detach().clone())
        return f"{cfg.PROMPT_LEN} soft-prompt vectors prepended to every pack"
    raise ValueError(f"unknown method {method}")


def count_params(model):
    every = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return every, train


def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().float().half() for k, v in model.state_dict().items() if k in names}


def layer_depth(name, n_layers):
    if ".layers." in name:
        return int(name.split(".layers.")[1].split(".")[0]) + 1
    if "emb" in name and name.startswith("backbone."):
        return 0
    return n_layers + 1


def build_optimizer(model, method, cfg):
    n_layers = len(model.backbone.layers)
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if not name.startswith("backbone.") and name != "prompt":
            lr = cfg.HEAD_LR                                    # BiLSTM / CRF / embeddings of the task head
        elif name == "prompt":
            lr = cfg.PROMPT_LR
        elif method in ("full", "kd"):
            lr = cfg.FULL_LR * cfg.LLRD ** (n_layers + 1 - layer_depth(name, n_layers))
        else:
            lr = cfg.ADAPTER_LR
        wd = 0.0 if (p.ndim < 2 or name == "prompt") else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()],
                             betas=(0.9, 0.98), eps=1e-6)


def class_weights(docs, power):
    counts = np.ones(N_LABELS)
    for d in docs:
        counts += np.bincount(d["labels"], minlength=N_LABELS)
    w = (counts.sum() / counts) ** power
    return torch.tensor(w / w.mean(), dtype=torch.float32), torch.tensor(np.log(counts / counts.sum()), dtype=torch.float32)


# --------------------------------------------------------------------------------------- metric
def macro_scores(gold, pred):
    """Macro P / R / F1 over the labels that occur in gold or prediction (sklearn's default behaviour)."""
    gold, pred = np.asarray(gold), np.asarray(pred)
    per = {}
    for lab in sorted(set(gold.tolist()) | set(pred.tolist())):
        tp = int(((gold == lab) & (pred == lab)).sum())
        p = tp / max(1, int((pred == lab).sum()))
        r = tp / max(1, int((gold == lab).sum()))
        per[LABELS[lab]] = {"p": p, "r": r, "f1": 2 * p * r / (p + r) if p + r else 0.0, "gold": int((gold == lab).sum())}
    macro = {k: float(np.mean([v[k] for v in per.values()])) for k in ("p", "r", "f1")}
    return macro, per


def top_confusions(gold, pred, k=8):
    pairs = {}
    for g, p in zip(gold, pred):
        if g != p:
            pairs[(LABELS[g], LABELS[p])] = pairs.get((LABELS[g], LABELS[p]), 0) + 1
    return sorted(pairs.items(), key=lambda kv: -kv[1])[:k]


# ------------------------------------------------------------------------------------ inference
@torch.no_grad()
def document_emissions(model, docs, pad_id, device, pack_batch):
    model.eval()
    out = []
    for d in docs:
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            _, em, _ = model(d["tok"], d["domain"], pad_id, device, pack_batch=pack_batch)
        out.append(F.log_softmax(em.float(), -1))
    return out


def decode(model, emissions, log_prior, decoder, tau):
    preds = []
    for em in emissions:
        adj = em - tau * log_prior.to(em.device)                # logit adjustment (Menon et al. 2021) for macro-F1
        preds.append(model.crf.viterbi(adj) if decoder == "crf" else adj.argmax(-1).tolist())
    return preds


def tune_decoding(model, docs, emissions, log_prior, taus):
    """Pick decoder (CRF / argmax) and prior-correction strength on the dev documents."""
    gold = [l for d in docs for l in d["labels"]]
    best = None
    grid = []
    for decoder in ("crf", "argmax"):
        for tau in taus:
            pred = [l for p in decode(model, emissions, log_prior, decoder, tau) for l in p]
            f1 = macro_scores(gold, pred)[0]["f1"]
            grid.append({"decoder": decoder, "tau": tau, "f1": f1})
            if best is None or f1 > best["f1"]:
                best = {"decoder": decoder, "tau": tau, "f1": f1}
    return best, grid


def write_submission(test_docs, preds, out_dir, zip_name):
    os.makedirs(out_dir, exist_ok=True)
    payload = {}
    for d, p in zip(test_docs, preds):
        assert len(p) == len(d["sents"]), f"{d['id']}: {len(p)} predictions for {len(d['sents'])} sentences"
        payload[d["id"]] = [LABELS[i] for i in p]
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    zip_path = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ["prediction.json"]
    return path, zip_path


def training_spans(docs, span, rng):
    """Cut every document into runs of `span` consecutive sentences, starting at a random offset each epoch."""
    spans = []
    for di, d in enumerate(docs):
        n = len(d["sents"])
        if n <= span:
            spans.append((di, 0, n))
            continue
        start = rng.randrange(0, span)
        cuts = [0] + list(range(start, n, span))
        cuts = sorted(set(c for c in cuts if c < n)) + [n]
        for a, b in zip(cuts[:-1], cuts[1:]):
            if b - a >= 2:
                spans.append((di, a, b))
    rng.shuffle(spans)
    return spans

In [ ]:
%%writefile /kaggle/working/phase3_arch2/run_rr.py
"""
One Phase-3 experiment = one call:

    python run_rr.py --method lorma

Trains on CL + IT together (a domain embedding tells them apart), keeps the epoch with the best dev macro-F1,
tunes the decoder (CRF vs argmax) and the prior correction on dev, writes the test predictions + submission zip
and pushes everything to Hugging Face.
"""
import os
import sys
import json
import time
import random
import argparse

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as C
import backbone as BB
from rr_core import (load_split, tokenize_docs, RRModel, configure_method, count_params, trainable_state,
                     build_optimizer, class_weights, macro_scores, top_confusions, document_emissions, decode,
                     tune_decoding, write_submission, training_spans, DOMAINS, LABELS)


def log(*a):
    print(*a, flush=True)


def build_model(method, keep_layers=None):
    model = RRModel(BB.build_backbone(keep_layers=keep_layers, bidirectional=C.BIDIRECTIONAL), C)
    note = configure_method(model, method, C, model.backbone.tok_emb.weight)
    return model, note


def load_finetuned(method, device):
    """Rebuild a model trained earlier by this notebook (the distillation teacher)."""
    path = os.path.join(C.OUT_DIR, method, f"rr_{method}.pt")
    if not os.path.exists(path):
        path = BB.fetch_from_hub(C.RR_REPO, f"{method}/rr_{method}.pt", os.path.join(C.OUT_DIR, "_hub"))
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"No fine-tuned '{method}' model locally or on {C.RR_REPO}; run '{method}' first.")
    blob = torch.load(path, map_location="cpu")
    model, _ = build_model(method, blob.get("keep_layers"))
    _, unexpected = model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    assert not unexpected, unexpected
    return model.to(device).eval()


def push(files, message):
    if not os.environ.get("HF_TOKEN"):
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(C.RR_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
        for local, remote in files:
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=C.RR_REPO,
                            token=os.environ["HF_TOKEN"], commit_message=message)
        log(f"pushed to {C.RR_REPO}: {message}")
    except Exception as e:
        log(f"HF push failed ({type(e).__name__}: {e}) - the files are still in {C.OUT_DIR}")


def evaluate(model, docs, pad_id, device, log_prior):
    ems = document_emissions(model, docs, pad_id, device, C.EVAL_PACK_BATCH)
    gold = [l for d in docs for l in d["labels"]]
    pred = [l for p in decode(model, ems, log_prior, "crf", 0.0) for l in p]
    return macro_scores(gold, pred)[0]["f1"], ems


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=["full", "lora", "lorma", "ia3", "prompt", "kd"])
    method = ap.parse_args().method

    random.seed(C.SEED)
    np.random.seed(C.SEED)
    torch.manual_seed(C.SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = os.path.join(C.OUT_DIR, method)
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()

    # ---------------------------------------------------------------- data
    tok = BB.load_tokenizer()
    train = [d for dom in DOMAINS for d in load_split(C.FILES[f"{dom}_train"], dom)]
    dev = [d for dom in DOMAINS for d in load_split(C.FILES[f"{dom}_dev"], dom)]
    test = [d for dom in DOMAINS for d in load_split(C.FILES[f"{dom}_test"], dom)]
    if C.TRAIN_ON_DEV:
        train = train + dev                                     # final run only: dev is then no longer a fair check
    if C.TRAIN_DOC_LIMIT:
        train = train[:C.TRAIN_DOC_LIMIT // 2] + train[-(C.TRAIN_DOC_LIMIT // 2):]
    for docs in (train, dev, test):
        tokenize_docs(docs, tok, C.MAX_SENT_TOKENS)
    log(f"[{method}] train docs {len(train)} ({sum(len(d['sents']) for d in train):,} sentences) | "
        f"dev docs {len(dev)} | test docs {len(test)}")

    weights, log_prior = class_weights(train, C.CLASS_WEIGHT_POWER)
    weights = weights.to(device)

    # --------------------------------------------------------------- model
    keep = BB.student_layers() if method == "kd" else None
    model, note = build_model(method, keep)
    model.to(device)
    total, trainable = count_params(model)
    log(f"[{method}] {note} | params {total / 1e6:.1f}M, trainable {trainable / 1e6:.2f}M ({100 * trainable / total:.2f}%)")

    teacher = None
    if method == "kd":
        teacher = load_finetuned(C.KD_TEACHER, device)
        log(f"[kd] teacher = fine-tuned '{C.KD_TEACHER}' ({count_params(teacher)[0] / 1e6:.1f}M) -> "
            f"student with backbone layers {keep} ({total / 1e6:.1f}M)")

    optimizer = build_optimizer(model, method, C)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    rng = random.Random(C.SEED)
    n_spans = len(training_spans(train, C.SPAN_SENTS, random.Random(0)))
    total_steps = -(-n_spans // C.GRAD_ACCUM) * C.EPOCHS
    warmup = max(1, int(C.WARMUP_FRAC * total_steps))

    best, stale, step, history = {"f1": -1.0}, 0, 0, []
    for epoch in range(1, C.EPOCHS + 1):
        model.train()
        spans = training_spans(train, C.SPAN_SENTS, rng)
        running, seen = 0.0, 0
        for i, (di, a, b) in enumerate(spans):
            d = train[di]
            gold = torch.tensor(d["labels"][a:b], device=device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                ctx, em, shift = model(d["tok"][a:b], d["domain"], tok.pad_id, device, n_total=len(d["sents"]), offset=a)
                loss = model.crf.nll(em, gold) / len(gold)
                loss = loss + C.CE_WEIGHT * F.cross_entropy(em, gold, weight=weights)
                if len(gold) > 1:
                    changed = (gold[1:] != gold[:-1]).float()
                    loss = loss + C.SHIFT_WEIGHT * F.binary_cross_entropy_with_logits(shift, changed)
                if teacher is not None:
                    with torch.no_grad():
                        t_ctx, t_em, _ = teacher(d["tok"][a:b], d["domain"], tok.pad_id, device, n_total=len(d["sents"]), offset=a)
                    T = C.KD_TEMPERATURE
                    kl = F.kl_div(F.log_softmax(em / T, -1), F.softmax(t_em.float() / T, -1), reduction="batchmean") * T * T
                    hid = F.mse_loss(ctx.float(), t_ctx.float())     # match the teacher's sentence representations too
                    loss = C.KD_ALPHA * loss + (1 - C.KD_ALPHA) * kl + C.KD_HIDDEN_WEIGHT * hid
            scaler.scale(loss / C.GRAD_ACCUM).backward()
            running += loss.item()
            seen += 1

            if seen % C.GRAD_ACCUM == 0 or i == len(spans) - 1:
                frac = (step + 1) / warmup if step < warmup else max(0.0, (total_steps - step) / max(1, total_steps - warmup))
                for g, lr in zip(optimizer.param_groups, base_lrs):
                    g["lr"] = lr * frac
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                step += 1

        f1, _ = evaluate(model, dev, tok.pad_id, device, log_prior)
        history.append({"epoch": epoch, "train_loss": running / max(1, seen), "dev_macro_f1": f1})
        log(f"[{method}] epoch {epoch}/{C.EPOCHS} | loss {running / max(1, seen):.4f} | dev macro-F1 (CRF) {f1:.4f} | "
            f"{(time.time() - t0) / 60:.1f} min")
        if f1 > best["f1"]:
            best = {"f1": f1, "epoch": epoch}
            torch.save({"method": method, "keep_layers": keep, "weights": trainable_state(model), "dev_f1": f1,
                        "epoch": epoch}, os.path.join(out_dir, f"rr_{method}.pt"))
            stale = 0
        else:
            stale += 1
            if stale >= C.PATIENCE:
                log(f"[{method}] no dev improvement for {C.PATIENCE} epochs - stopping early")
                break

    # ------------------------------------------- best epoch: tune decoding on dev, predict test
    blob = torch.load(os.path.join(out_dir, f"rr_{method}.pt"), map_location="cpu")
    model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    t_inf = time.time()
    dev_ems = document_emissions(model, dev, tok.pad_id, device, C.EVAL_PACK_BATCH)
    dev_seconds = time.time() - t_inf
    choice, grid = tune_decoding(model, dev, dev_ems, log_prior, C.TAU_GRID)
    dev_pred = decode(model, dev_ems, log_prior, choice["decoder"], choice["tau"])

    gold_all = [l for d in dev for l in d["labels"]]
    pred_all = [l for p in dev_pred for l in p]
    macro, per = macro_scores(gold_all, pred_all)
    by_domain = {}
    for k, dom in enumerate(DOMAINS):
        g = [l for d in dev if d["domain"] == k for l in d["labels"]]
        p = [l for d, pr in zip(dev, dev_pred) if d["domain"] == k for l in pr]
        by_domain[dom] = macro_scores(g, p)[0]

    test_ems = document_emissions(model, test, tok.pad_id, device, C.EVAL_PACK_BATCH)
    test_pred = decode(model, test_ems, log_prior, choice["decoder"], choice["tau"])
    _, zip_path = write_submission(test, test_pred, out_dir, f"submission_{C.ARCH}_{method}.zip")

    summary = {
        "arch": C.ARCH, "method": method, "best_epoch": best["epoch"], "decoder": choice["decoder"], "tau": choice["tau"],
        "dev_macro": macro, "dev_by_domain": by_domain, "dev_per_label": per, "decoding_grid": grid,
        "confusions": [[g, p, n] for (g, p), n in top_confusions(gold_all, pred_all)],
        "params_total": total, "params_trainable": trainable, "dev_inference_seconds": round(dev_seconds, 2),
        "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if device.type == "cuda" else 0.0,
        "train_minutes": round((time.time() - t0) / 60, 1), "history": history,
        "trained_on_dev": bool(C.TRAIN_ON_DEV),
    }
    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(summary, f, indent=2)
    log(f"[{method}] DONE | dev macro-F1 {macro['f1']:.4f} (CL {by_domain['CL']['f1']:.4f}, IT {by_domain['IT']['f1']:.4f}) "
        f"| decoder {choice['decoder']}, tau {choice['tau']} | epoch {best['epoch']} -> {zip_path}")

    push([(os.path.join(out_dir, f"rr_{method}.pt"), f"{method}/rr_{method}.pt"),
          (os.path.join(out_dir, "results.json"), f"{method}/results.json"),
          (zip_path, f"{method}/{os.path.basename(zip_path)}")], f"{C.ARCH} {method}: dev macro-F1 {macro['f1']:.4f}")


if __name__ == "__main__":
    main()

## 4. Phase-3 data

In [ ]:
import importlib
import config as C
importlib.reload(C)

for dom in ("CL", "IT"):
    for name in (f"Train/{dom}_train.json", f"Train/{dom}_dev.json", f"Test/{dom}_test.json"):
        hf_hub_download(C.HF_DATASET_REPO, f"Phase_3_RR/{name}", repo_type="dataset", local_dir=C.DATA_DIR, token=HF_TOKEN)
hf_hub_download(C.HF_DATASET_REPO, "Phase_3_RR/Train/label_vocab.json", repo_type="dataset", local_dir=C.DATA_DIR, token=HF_TOKEN)

import rr_core as R
importlib.reload(R)
vocab = json.load(open(os.path.join(C.DATA_DIR, "Phase_3_RR/Train/label_vocab.json")))
assert sorted(vocab, key=vocab.get) == R.LABELS, "label vocabulary changed - update LABELS in rr_core.py"

from collections import Counter
for dom in R.DOMAINS:
    tr, dv, te = (R.load_split(C.FILES[f"{dom}_{s}"], dom) for s in ("train", "dev", "test"))
    counts = Counter(R.LABELS[l] for d in tr for l in d["labels"])
    sticky = sum(a == b for d in tr for a, b in zip(d["labels"], d["labels"][1:])) / sum(len(d["labels"]) - 1 for d in tr)
    print(f"{dom}: train {len(tr)} docs / {sum(len(d['sents']) for d in tr):,} sentences | dev {len(dv)} | test {len(te)} docs "
          f"| same role as previous sentence: {sticky:.0%}")
    print("   ", dict(counts.most_common()))

## 5. Load the Phase-1 weights once
This checks the model and caches the files for the experiment processes.

In [ ]:
import backbone as BB
importlib.reload(BB)
tok = BB.load_tokenizer()
probe = BB.build_backbone()
sample = R.tokenize_docs(R.load_split(C.FILES["CL_dev"], "CL")[:1], tok, C.MAX_SENT_TOKENS)[0]
packs, where = R.pack(sample["tok"][:40], C.PACK_TOKENS)
ids, mask, _ = R.batch_packs(packs, where, tok.pad_id, torch.device("cpu"))
with torch.no_grad():
    h = probe(ids, mask)
print(f"backbone ready: {sum(p.numel() for p in probe.parameters()) / 1e6:.1f}M parameters | "
      f"40 sentences -> {len(packs)} packs -> hidden states {tuple(h.shape)}")
del probe, h

## 6. Run the experiments
Each experiment is its own process (`run_rr.py --method ...`); with two GPUs, two run at once.
`kd` starts as soon as `full` (its teacher) is done. Results, checkpoints and zips are pushed to your HF repo as they finish.

In [ ]:
import subprocess, threading

pending = list(C.METHODS)
done, failed = set(), set()
lock = threading.Lock()


def next_job():
    with lock:
        if "kd" in pending and C.KD_TEACHER in failed:
            pending.remove("kd")
            print(f"skipping kd: its teacher '{C.KD_TEACHER}' failed")
        for m in pending:
            if m != "kd" or C.KD_TEACHER in done or C.KD_TEACHER not in C.METHODS:
                pending.remove(m)
                return m
        return "wait" if pending else None


def worker(gpu):
    while True:
        job = next_job()
        if job is None:
            return
        if job == "wait":
            time.sleep(15)
            continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        proc = subprocess.Popen([sys.executable, "run_rr.py", "--method", job], cwd=WORK_DIR, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(f"[gpu{gpu}] {line}", end="")
        with lock:
            (done if proc.wait() == 0 else failed).add(job)


t0 = time.time()
threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"\nfinished in {(time.time() - t0) / 60:.0f} min | ok: {sorted(done)} | failed: {sorted(failed)}")

## 7. Results
Dev macro-F1 overall and per domain, cost of each method, per-role scores and the most frequent confusions.

In [ ]:
import pandas as pd

rows, per_label, confusions = [], {}, {}
for path in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "results.json"))):
    r = json.load(open(path))
    m = r["method"]
    rows.append({"method": m, "dev macro-F1": r["dev_macro"]["f1"], "CL F1": r["dev_by_domain"]["CL"]["f1"],
                 "IT F1": r["dev_by_domain"]["IT"]["f1"], "macro P": r["dev_macro"]["p"], "macro R": r["dev_macro"]["r"],
                 "decoder": r["decoder"], "tau": r["tau"], "best epoch": r["best_epoch"],
                 "params (M)": r["params_total"] / 1e6, "trainable %": 100 * r["params_trainable"] / r["params_total"],
                 "train min": r["train_minutes"], "peak GPU GB": r["peak_gpu_gb"], "dev infer s": r["dev_inference_seconds"]})
    per_label[m] = {lab: v["f1"] for lab, v in r["dev_per_label"].items()}
    confusions[m] = r["confusions"]

table = pd.DataFrame(rows).sort_values("dev macro-F1", ascending=False).round(4)
display(table)
display(pd.DataFrame(per_label).round(3))
best = table.iloc[0]["method"]
print(f"most frequent confusions for '{best}' (gold -> predicted, count):")
for g, p, n in confusions[best]:
    print(f"   {g:24s} -> {p:24s} {n}")

## 8. Pick a submission
Every experiment wrote `runs/<method>/submission_<arch>_<method>.zip`: a single `prediction.json` at the root with one
label per sentence for all CL and IT test documents. The best on dev is also copied to `/kaggle/working/submission.zip`.

In [ ]:
src = glob.glob(os.path.join(C.OUT_DIR, best, "submission_*.zip"))[0]
shutil.copy(src, "/kaggle/working/submission.zip")
print(f"best on dev: {best} -> /kaggle/working/submission.zip")
for z in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "submission_*.zip"))):
    print("  ", z)